In [0]:

from pyspark.sql import functions as F

# Read data from Bronze layer
orders_df = spark.table("olist.bronze.orders")

# Display first 10 records
display(orders_df.limit(10))

# Check total rows
print("Total orders:", orders_df.count())


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,delivered,2017-07-09 21:57:05,2017-07-09 22:10:13,2017-07-11 14:58:04,2017-07-26 10:57:55,2017-08-01 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,invoiced,2017-04-11 12:22:08,2017-04-13 13:25:17,null,null,2017-05-09 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,delivered,2017-05-16 13:10:30,2017-05-16 13:22:11,2017-05-22 10:07:46,2017-05-26 12:55:51,2017-06-07 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,delivered,2017-01-23 18:29:09,2017-01-25 02:50:47,2017-01-26 14:16:31,2017-02-02 14:08:10,2017-03-06 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,delivered,2017-07-29 11:55:02,2017-07-29 12:05:32,2017-08-10 19:45:24,2017-08-16 17:14:30,2017-08-23 00:00:00,2026-11-02T10:24:45.259Z,olist_orders_dataset.csv


Total orders: 99441


In [0]:

# Check missing values in each column

null_counts = orders_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in orders_df.columns
])

display(null_counts)


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,ingestion_timestamp,source_file
0,0,0,0,160,1783,2965,0,0,0


In [0]:

# Step 4.4: Check and remove duplicate orders

print("Rows before cleaning:", orders_df.count())

duplicate_count = (
    orders_df.count()
    - orders_df.select("order_id").distinct().count()
)

print("Duplicate order IDs:", duplicate_count)

# Remove duplicate order records
orders_clean = orders_df.dropDuplicates(["order_id"])

print("Rows after cleaning:", orders_clean.count())


Rows before cleaning: 99441
Duplicate order IDs: 0
Rows after cleaning: 99441


In [0]:

# Step 4.5: Convert string dates to timestamps

date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders_clean = orders_clean.withColumn(
        column,
        F.to_timestamp(F.col(column))
    )

# Check converted data types
orders_clean.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# Step 4.6: Calculate delivery delays

orders_clean = orders_clean.withColumn(
    "delivery_delay_days",
    F.when(
        F.col("order_delivered_customer_date").isNotNull(),
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date")
        )
    )
)

# Classify delivery status
orders_clean = orders_clean.withColumn(
    "delivery_status",
    F.when(
        F.col("order_delivered_customer_date").isNull(),
        "Not Delivered"
    )
    .when(
        F.col("delivery_delay_days") > 0,
        "Late"
    )
    .otherwise("On Time")
)

display(
    orders_clean.select(
        "order_id",
        "order_status",
        "delivery_delay_days",
        "delivery_status"
    ).limit(10)
)


order_id,order_status,delivery_delay_days,delivery_status
e481f51cbdc54678b7cc49136f2d6af7,delivered,-8,On Time
53cdb2fc8bc7dce0b6741e2150273451,delivered,-6,On Time
47770eb9100c2d0c44946d9cf07ec65d,delivered,-18,On Time
949d5b44dbf5de918fe9c16f97b45f8a,delivered,-13,On Time
ad21c59c0840e6cb83a9ceb5573f8159,delivered,-10,On Time
a4591c265e18cb1dcee52889e2d8acc3,delivered,-6,On Time
136cce7faa42fdb2cefd53fdc79a6098,invoiced,null,Not Delivered
6514b8ad8028c9f2cc2374ded245783f,delivered,-12,On Time
76c6e866289321a7c93b82b54852dc33,delivered,-32,On Time
e69bfb5eb88e0ed6a785585b27e16dbf,delivered,-7,On Time


In [0]:

# Count orders by delivery status

display(
    orders_clean.groupBy("delivery_status")
    .count()
    .orderBy("delivery_status")
)


delivery_status,count
Late,6535
Not Delivered,2965
On Time,89941


In [0]:

# Save transformed orders data

(
    orders_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.orders")
)

print("Silver Orders table created successfully!")


Silver Orders table created successfully!


In [0]:

# Step 4.9: Load Bronze Order Items

from pyspark.sql import functions as F

items_df = spark.table("olist.bronze.order_items")

print("Total order items:", items_df.count())

display(items_df.limit(10))


Total order items: 112650


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,source_file
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40,2026-11-02T10:25:19.364Z,olist_order_items_dataset.csv


In [0]:

# Check missing values in Order Items

null_counts = items_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in items_df.columns
])

display(null_counts)


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,source_file
0,0,0,0,0,0,0,0,0


In [0]:

# Check duplicate order items

total_rows = items_df.count()

unique_items = items_df.select(
    "order_id",
    "order_item_id"
).distinct().count()

print("Total rows:", total_rows)
print("Unique order items:", unique_items)
print("Duplicate records:", total_rows - unique_items)

# Remove duplicate item keys
items_clean = items_df.dropDuplicates(
    ["order_id", "order_item_id"]
)

print("Rows after cleaning:", items_clean.count())


Total rows: 112650
Unique order items: 112650
Duplicate records: 0
Rows after cleaning: 112650


In [0]:

# Convert columns to appropriate data types

items_clean = (
    items_clean
    .withColumn(
        "order_item_id",
        F.col("order_item_id").cast("int")
    )
    .withColumn(
        "price",
        F.col("price").cast("double")
    )
    .withColumn(
        "freight_value",
        F.col("freight_value").cast("double")
    )
    .withColumn(
        "shipping_limit_date",
        F.to_timestamp("shipping_limit_date")
    )
)

items_clean.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# Calculate item total including freight

items_clean = items_clean.withColumn(
    "item_total_value",
    F.col("price") + F.col("freight_value")
)

display(
    items_clean.select(
        "order_id",
        "product_id",
        "price",
        "freight_value",
        "item_total_value"
    ).limit(10)
)


order_id,product_id,price,freight_value,item_total_value
00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,58.9,13.29,72.19
00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,239.9,19.93,259.83
000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,199.0,17.87,216.87
00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,12.99,12.79,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,199.9,18.14,218.04000000000002
00048cc3ae777c65dbb7d2a0634bc1ea,ef92defde845ab8450f9d70c526ef70f,21.9,12.69,34.589999999999996
00054e8431b9d7675808bcb819fb4a32,8d4f2bb7e93e6710a28f34fa83ee7d28,19.9,11.85,31.75
000576fe39319847cbb9d288c5617fa6,557d850972a7d6f792fd18ae1400d9b6,810.0,70.75,880.75
0005a1a1728c9d785b8e2b08b904576c,310ae3c140ff94b03219ad0adc3c778f,145.95,11.65,157.6
0005f50442cb953dcd1d21e1fb923495,4535b0e1091c278dfd193e5a1d63b39f,53.99,11.4,65.39


In [0]:

# Check invalid numeric values

invalid_items = items_clean.filter(
    F.col("price").isNull()
    | (F.col("price") < 0)
    | F.col("freight_value").isNull()
    | (F.col("freight_value") < 0)
)

print("Invalid price/freight records:",
      invalid_items.count())

display(invalid_items.limit(10))


Invalid price/freight records: 0


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,ingestion_timestamp,source_file,item_total_value


In [0]:

# Save transformed data in Silver layer

(
    items_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.order_items")
)

print("Silver Order Items table created successfully!")

print(
    "Total Silver records:",
    spark.table("olist.silver.order_items").count()
)


Silver Order Items table created successfully!
Total Silver records: 112650


In [0]:

display(
    spark.sql("""
        SELECT
            order_id,
            order_item_id,
            product_id,
            seller_id,
            price,
            freight_value,
            item_total_value
        FROM olist.silver.order_items
        LIMIT 10
    """)
)


order_id,order_item_id,product_id,seller_id,price,freight_value,item_total_value
0010b2e5201cc5f1ae7e9c6cc8f5bd00,1,5a419dbf24a8c9718fe522b81c69f61a,3504c0cb71d7fa48d967e0e4c94d59d9,48.9,16.6,65.5
0011d82c4b53e22e84023405fb467e57,1,c389f712c4b4510bc997cee93e8b1a28,bfd27a966d91cfaafdb25d076585f0da,289.0,26.33,315.33
00143d0f86d6fbd9f9b38ab440ac16f5,2,e95ee6822b66ac6058e2e4aff656071a,a17f621c590ea0fab3d5d883e1630ec6,21.33,15.1,36.43
0014ae671de39511f7575066200733b7,1,23365beed316535b4105bd800c46670e,92eb0f42c21942b6552362b9b114707d,16.5,14.1,30.6
0016dfedd97fc2950e388d2971d718c7,1,4089861a1bd4685da70bddd6b4f974f1,a35124e2d763d7ca3fbe3b97d143200f,49.75,20.8,70.55
001ab0a7578dd66cd4b0a71f5b6e1e41,2,0b0172eb0fd18479d29c3bc122c058c2,5656537e588803a555b8eb41f07a944b,24.89,17.63,42.519999999999996
002175704e8b209f61b9ad5cfd92b60e,1,e6b6e13cf71449a457269f425b89dc74,b2ba3715d723d245138f291a6fe42594,109.9,13.21,123.11000000000001
002d040018d12a3853c059f7f23ab5b1,1,97c58626cc3e15d514d2095f69650220,fe2032dab1a61af8794248c8196565c9,155.0,14.84,169.84
003423b755b562962a6225a8de40d12e,1,84904413b82eee333a3f79137e8d197e,23c38debaffe4a25a30fdbd9b586a13f,232.75,28.58,261.33
0035246a40f520710769010f752e7507,1,8a6187b2665118d5095f99a25fd7ba7a,4a3ca9315b744ce9f8e9374361493884,87.0,12.11,99.11


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Load Bronze Order Reviews table
reviews_df = spark.table("olist.bronze.order_reviews")

print("Total review records:", reviews_df.count())

display(reviews_df.limit(10))


Total review records: 104162


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,null,null,2018-01-18 00:00:00,2018-01-18 21:46:59,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,null,null,2018-03-10 00:00:00,2018-03-11 03:05:13,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,null,null,2018-02-17 00:00:00,2018-02-18 14:36:24,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,null,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,null,Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa,2018-03-01 00:00:00,2018-03-02 10:26:53,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,null,null,2018-04-13 00:00:00,2018-04-16 00:39:37,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,null,null,2017-07-16 00:00:00,2017-07-18 19:30:34,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,null,null,2018-08-14 00:00:00,2018-08-14 21:36:06,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,null,null,2017-05-17 00:00:00,2017-05-18 12:05:37,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,recomendo,aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho,2018-05-22 00:00:00,2018-05-23 16:45:47,2026-11-02T10:25:28.658Z,olist_order_reviews_dataset.csv


In [0]:

# Check missing values in reviews dataset

null_counts = reviews_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in reviews_df.columns
])

display(null_counts)


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file
1,2236,2380,92157,63079,8764,8785,0,0


In [0]:

# Check duplicate review IDs and multiple reviews per order

print("Total rows:", reviews_df.count())

print(
    "Duplicate review IDs:",
    reviews_df.count()
    - reviews_df.select("review_id").distinct().count()
)

print(
    "Additional reviews for repeated order IDs:",
    reviews_df.count()
    - reviews_df.select("order_id").distinct().count()
)

display(
    reviews_df.groupBy("order_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"))
    .limit(10)
)


Total rows: 104162
Duplicate review IDs: 1204
Additional reviews for repeated order IDs: 4419


order_id,count
null,2236
2018-05-15 00:00:00,16
2017-12-19 00:00:00,16
2017-12-16 00:00:00,15
2018-02-24 00:00:00,15
2018-04-11 00:00:00,14
2017-12-06 00:00:00,14
2018-08-18 00:00:00,14
2018-08-28 00:00:00,13
2018-08-17 00:00:00,12


In [0]:

# Convert review score and timestamps

reviews_clean = (
    reviews_df
    .withColumn(
        "review_score",
        F.col("review_score").cast("int")
    )
    .withColumn(
        "review_creation_date",
        F.to_timestamp("review_creation_date")
    )
    .withColumn(
        "review_answer_timestamp",
        F.to_timestamp("review_answer_timestamp")
    )
)

reviews_clean.printSchema()


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# Inspect the original Bronze review data

reviews_df = spark.table("olist.bronze.order_reviews")

display(
    reviews_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ).limit(10)
)


review_id,order_id,review_score,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,2018-05-22 00:00:00,2018-05-23 16:45:47


In [0]:

reviews_df.printSchema()


root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# Identify non-numeric review scores

display(
    reviews_df.filter(
        F.expr("try_cast(review_score AS INT)").isNull()
    ).select(
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date"
    ).limit(20)
)


review_id,order_id,review_score,review_creation_date
",2018-02-16 00:00:00,2018-02-20 10:52:22",null,null,null
A entrega foi efetuada muito antes do prazo dado.,null,null,null
O produto já começou a ser usado e até o presente,null,null,null
"sem problemas.""",2018-03-23 00:00:00,2018-04-01 00:27:51,null
"Faltou Controle Remoto para Ar Condicionado Consul""",2018-01-28 00:00:00,2018-02-04 11:06:15,null
Estou satisfeita,null,null,null
",2017-06-02 00:00:00,2017-06-03 13:12:07",null,null,null
",2017-08-09 00:00:00,2017-08-09 12:03:45",null,null,null
",2017-11-22 00:00:00,2017-11-22 14:45:40",null,null,null
"Muito obrigada.... """,2017-06-24 00:00:00,2017-06-29 13:48:21,null


In [0]:

reviews_fixed = (
    spark.read
    .option("header", "true")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "FAILFAST")
    .csv(
        "/Volumes/olist/bronze/raw_files/"
        "olist_order_reviews_dataset.csv"
    )
)

display(
    reviews_fixed.select(
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date"
    ).limit(10)
)


review_id,order_id,review_score,review_creation_date
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,2018-04-13 00:00:00
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,2017-07-16 00:00:00
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,2018-08-14 00:00:00
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,2017-05-17 00:00:00
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,2018-05-22 00:00:00


In [0]:

from pyspark.sql import functions as F

# Check invalid review scores in the repaired dataset
invalid_count = reviews_fixed.filter(
    F.expr("try_cast(review_score AS INT)").isNull()
    | ~F.expr("try_cast(review_score AS INT)").between(1, 5)
).count()

print("Total review records:", reviews_fixed.count())
print("Invalid review scores:", invalid_count)


Total review records: 99224
Invalid review scores: 0


In [0]:

# Add Bronze ingestion metadata
reviews_bronze_fixed = (
    reviews_fixed
    .withColumn(
        "ingestion_timestamp",
        F.current_timestamp()
    )
    .withColumn(
        "source_file",
        F.lit("olist_order_reviews_dataset.csv")
    )
)

# Replace incorrectly parsed Bronze table
(
    reviews_bronze_fixed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("olist.bronze.order_reviews")
)

print("Bronze Reviews table repaired successfully!")


Bronze Reviews table repaired successfully!


In [0]:

reviews_df = spark.table("olist.bronze.order_reviews")

reviews_clean = (
    reviews_df
    .withColumn(
        "review_score",
        F.col("review_score").cast("int")
    )
    .withColumn(
        "review_creation_date",
        F.to_timestamp("review_creation_date")
    )
    .withColumn(
        "review_answer_timestamp",
        F.to_timestamp("review_answer_timestamp")
    )
)

display(
    reviews_clean.select(
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date"
    ).limit(10)
)


review_id,order_id,review_score,review_creation_date
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18T00:00:00.000Z
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10T00:00:00.000Z
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17T00:00:00.000Z
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21T00:00:00.000Z
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01T00:00:00.000Z
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,2018-04-13T00:00:00.000Z
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,2017-07-16T00:00:00.000Z
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,2018-08-14T00:00:00.000Z
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,2017-05-17T00:00:00.000Z
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,2018-05-22T00:00:00.000Z


In [0]:

invalid_reviews = reviews_clean.filter(
    F.col("review_score").isNull()
    | ~F.col("review_score").between(1, 5)
)

print("Invalid review scores:", invalid_reviews.count())

display(invalid_reviews.limit(10))


Invalid review scores: 0


review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,ingestion_timestamp,source_file


In [0]:

from pyspark.sql import functions as F

# Load Products dataset
products_df = spark.table("olist.bronze.products")

print("Total Products:", products_df.count())

display(products_df.limit(10))


Total Products: 32951


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp,source_file
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,60,745,1,200,38,5,11,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
732bd381ad09e530fe0a5f457d81becb,cool_stuff,56,1272,4,18350,70,24,44,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,56,184,2,900,40,8,40,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
37cc742be07708b53a98702e77a21a02,eletrodomesticos,57,163,1,400,27,13,17,2026-11-02T10:25:33.080Z,olist_products_dataset.csv
8c92109888e8cdf9d66dc7e463025574,brinquedos,36,1156,1,600,17,10,12,2026-11-02T10:25:33.080Z,olist_products_dataset.csv


In [0]:

# Check missing values in products dataset

null_counts = products_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in products_df.columns
])

display(null_counts)


product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp,source_file
0,610,610,610,610,2,2,2,2,0,0


In [0]:

# Check duplicate product IDs

total_products = products_df.count()

unique_products = products_df.select(
    "product_id"
).distinct().count()

print("Total products:", total_products)
print("Unique product IDs:", unique_products)
print("Duplicate product IDs:",
      total_products - unique_products)

# View duplicated product IDs
display(
    products_df.groupBy("product_id")
    .count()
    .filter(F.col("count") > 1)
    .limit(10)
)


Total products: 32951
Unique product IDs: 32951
Duplicate product IDs: 0


product_id,count


In [0]:

# Convert product measurements to numeric types

numeric_columns = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

products_clean = products_df

for column in numeric_columns:
    products_clean = products_clean.withColumn(
        column,
        F.expr(f"try_cast({column} AS DOUBLE)")
    )

products_clean.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: double (nullable = true)
 |-- product_description_lenght: double (nullable = true)
 |-- product_photos_qty: double (nullable = true)
 |-- product_weight_g: double (nullable = true)
 |-- product_length_cm: double (nullable = true)
 |-- product_height_cm: double (nullable = true)
 |-- product_width_cm: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# Calculate product volume in cubic centimeters

products_clean = products_clean.withColumn(
    "product_volume_cm3",
    F.col("product_length_cm")
    * F.col("product_height_cm")
    * F.col("product_width_cm")
)

display(
    products_clean.select(
        "product_id",
        "product_weight_g",
        "product_volume_cm3"
    ).limit(10)
)


product_id,product_weight_g,product_volume_cm3
1e9e8ef04dbcff4541ed26657ea517e5,225.0,2240.0
3aa071139cb16b67ca9e5dea641aaa2f,1000.0,10800.0
96bd76ec8810374ed1b65e291975717f,154.0,2430.0
cef67bcfe19066a932b7673e239eb23d,371.0,2704.0
9dc1a7de274444849c219cff195d0b71,625.0,4420.0
41d3672d4792049fa1779bb35283ed13,200.0,2090.0
732bd381ad09e530fe0a5f457d81becb,18350.0,73920.0
2548af3e6e77a690cf3eb6368e9ab61e,900.0,12800.0
37cc742be07708b53a98702e77a21a02,400.0,5967.0
8c92109888e8cdf9d66dc7e463025574,600.0,2040.0


In [0]:

# Load category translation table

translations_df = spark.table(
    "olist.bronze.category_translation"
)

display(translations_df.limit(10))


product_category_name,product_category_name_english,ingestion_timestamp,source_file
beleza_saude,health_beauty,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
informatica_acessorios,computers_accessories,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
automotivo,auto,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
cama_mesa_banho,bed_bath_table,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
moveis_decoracao,furniture_decor,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
esporte_lazer,sports_leisure,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
perfumaria,perfumery,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
utilidades_domesticas,housewares,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
telefonia,telephony,2026-11-02T10:25:47.382Z,product_category_name_translation.csv
relogios_presentes,watches_gifts,2026-11-02T10:25:47.382Z,product_category_name_translation.csv


In [0]:

display(
    translations_df.groupBy("product_category_name")
    .count()
    .filter(F.col("count") > 1)
)


product_category_name,count


In [0]:

# Clean translation table

translations_clean = (
    translations_df
    .select(
        "product_category_name",
        "product_category_name_english"
    )
    .dropDuplicates(["product_category_name"])
)

# Join products with English categories
products_enriched = products_clean.join(
    translations_clean,
    on="product_category_name",
    how="left"
)

# Use Portuguese category if translation is unavailable
products_enriched = products_enriched.withColumn(
    "category_english",
    F.coalesce(
        F.col("product_category_name_english"),
        F.col("product_category_name"),
        F.lit("Unknown")
    )
)

display(
    products_enriched.select(
        "product_id",
        "product_category_name",
        "category_english"
    ).limit(10)
)


product_id,product_category_name,category_english
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
3aa071139cb16b67ca9e5dea641aaa2f,artes,art
96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
cef67bcfe19066a932b7673e239eb23d,bebes,baby
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares
41d3672d4792049fa1779bb35283ed13,instrumentos_musicais,musical_instruments
732bd381ad09e530fe0a5f457d81becb,cool_stuff,cool_stuff
2548af3e6e77a690cf3eb6368e9ab61e,moveis_decoracao,furniture_decor
37cc742be07708b53a98702e77a21a02,eletrodomesticos,home_appliances
8c92109888e8cdf9d66dc7e463025574,brinquedos,toys


In [0]:

# Identify negative or zero measurements

invalid_products = products_enriched.filter(
    (F.col("product_weight_g") <= 0)
    | (F.col("product_length_cm") <= 0)
    | (F.col("product_height_cm") <= 0)
    | (F.col("product_width_cm") <= 0)
)

print(
    "Products with invalid measurements:",
    invalid_products.count()
)

display(invalid_products.limit(10))


Products with invalid measurements: 4


product_category_name,product_id,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,ingestion_timestamp,source_file,product_volume_cm3,product_category_name_english,category_english
cama_mesa_banho,81781c0fed9fe1ad6e8c81fca1e1cb08,51.0,529.0,1.0,0.0,30.0,25.0,30.0,2026-11-02T10:25:33.080Z,olist_products_dataset.csv,22500.0,bed_bath_table,bed_bath_table
cama_mesa_banho,8038040ee2a71048d4bdbbdc985b69ab,48.0,528.0,1.0,0.0,30.0,25.0,30.0,2026-11-02T10:25:33.080Z,olist_products_dataset.csv,22500.0,bed_bath_table,bed_bath_table
cama_mesa_banho,36ba42dd187055e1fbe943b2d11430ca,53.0,528.0,1.0,0.0,30.0,25.0,30.0,2026-11-02T10:25:33.080Z,olist_products_dataset.csv,22500.0,bed_bath_table,bed_bath_table
cama_mesa_banho,e673e90efa65a5409ff4196c038bb5af,53.0,528.0,1.0,0.0,30.0,25.0,30.0,2026-11-02T10:25:33.080Z,olist_products_dataset.csv,22500.0,bed_bath_table,bed_bath_table


In [0]:

(
    translations_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.category_translation")
)


In [0]:

(
    products_enriched.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.products")
)

print("Silver Products table created successfully!")

print(
    "Total Silver products:",
    spark.table("olist.silver.products").count()
)


Silver Products table created successfully!
Total Silver products: 32951


In [0]:

# Check product counts by English category

display(
    spark.table("olist.silver.products")
    .groupBy("category_english")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
)


category_english,count
bed_bath_table,3029
sports_leisure,2867
furniture_decor,2657
health_beauty,2444
housewares,2335
auto,1900
computers_accessories,1639
toys,1411
watches_gifts,1329
telephony,1134


In [0]:

from pyspark.sql import functions as F

# Load Bronze Customers table
customers_df = spark.table("olist.bronze.customers")

print("Total customer records:", customers_df.count())

display(customers_df.limit(10))


Total customer records: 99441


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,source_file
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv


In [0]:

# Check missing values in customer records

customer_nulls = customers_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in customers_df.columns
])

display(customer_nulls)


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,source_file
0,0,0,0,0,0,0


In [0]:

# Check duplicate customer IDs

customer_duplicates = (
    customers_df.groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate customer IDs:", customer_duplicates.count())

display(customer_duplicates.limit(10))


Duplicate customer IDs: 0


customer_id,count


In [0]:

# Standardize customer location fields

customers_clean = (
    customers_df
    .withColumn(
        "customer_city",
        F.lower(F.trim(F.col("customer_city")))
    )
    .withColumn(
        "customer_state",
        F.upper(F.trim(F.col("customer_state")))
    )
    .withColumn(
        "customer_zip_code_prefix",
        F.lpad(
            F.trim(F.col("customer_zip_code_prefix")),
            5,
            "0"
        )
    )
)

display(customers_clean.limit(10))


customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,ingestion_timestamp,source_file
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,08775,mogi das cruzes,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,04534,sao paulo,SP,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv
4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,2026-11-02T10:25:14.639Z,olist_customers_dataset.csv


In [0]:

# Customers by state

display(
    customers_clean.groupBy("customer_state")
    .count()
    .orderBy(F.desc("count"))
)


customer_state,count
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033
GO,2020


In [0]:

# Save cleaned Customers table

(
    customers_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.customers")
)

print("Silver Customers table created successfully!")


Silver Customers table created successfully!


In [0]:

# Load Bronze Sellers table

sellers_df = spark.table("olist.bronze.sellers")

print("Total seller records:", sellers_df.count())

display(sellers_df.limit(10))


Total seller records: 3095


seller_id,seller_zip_code_prefix,seller_city,seller_state,ingestion_timestamp,source_file
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv


In [0]:

# Check missing values in seller records

seller_nulls = sellers_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in sellers_df.columns
])

display(seller_nulls)


seller_id,seller_zip_code_prefix,seller_city,seller_state,ingestion_timestamp,source_file
0,0,0,0,0,0


In [0]:

# Check duplicate seller IDs

seller_duplicates = (
    sellers_df.groupBy("seller_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate seller IDs:", seller_duplicates.count())

display(seller_duplicates.limit(10))


Duplicate seller IDs: 0


seller_id,count


In [0]:

# Standardize seller location data

sellers_clean = (
    sellers_df
    .withColumn(
        "seller_city",
        F.lower(F.trim(F.col("seller_city")))
    )
    .withColumn(
        "seller_state",
        F.upper(F.trim(F.col("seller_state")))
    )
    .withColumn(
        "seller_zip_code_prefix",
        F.lpad(
            F.trim(F.col("seller_zip_code_prefix")),
            5,
            "0"
        )
    )
)

display(sellers_clean.limit(10))


seller_id,seller_zip_code_prefix,seller_city,seller_state,ingestion_timestamp,source_file
3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
c0f3eea2e14555b6faeea3dd58c1b1c3,04195,sao paulo,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
c240c4061717ac1806ae6ee72be3533b,20920,rio de janeiro,RJ,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
e49c26c3edfa46d227d5121a6b6e4d37,55325,brejao,PE,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
1b938a7ec6ac5061a66a3766e0e75f90,16304,penapolis,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
768a86e36ad6aae3d03ee3c6433d61df,01529,sao paulo,SP,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv
ccc4bbb5f32a6ab2b7066a4130f114e3,80310,curitiba,PR,2026-11-02T10:25:37.607Z,olist_sellers_dataset.csv


In [0]:

# Distribution of sellers by state

display(
    sellers_clean.groupBy("seller_state")
    .count()
    .orderBy(F.desc("count"))
)


seller_state,count
SP,1849
PR,349
MG,244
SC,190
RJ,171
RS,129
GO,40
DF,30
ES,23
BA,19


In [0]:

# Save cleaned Sellers table

(
    sellers_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.sellers")
)

print("Silver Sellers table created successfully!")


Silver Sellers table created successfully!


In [0]:

# Verify Customers and Sellers Silver tables

display(
    spark.sql("""
        SELECT 'customers' AS table_name,
               COUNT(*) AS total_records
        FROM olist.silver.customers

        UNION ALL

        SELECT 'sellers' AS table_name,
               COUNT(*) AS total_records
        FROM olist.silver.sellers
    """)
)


table_name,total_records
customers,99441
sellers,3095


In [0]:

from pyspark.sql import functions as F

# Load Bronze Order Payments
payments_df = spark.table("olist.bronze.order_payments")

print("Total payment records:", payments_df.count())

display(payments_df.limit(10))


Total payment records: 103886


order_id,payment_sequential,payment_type,payment_installments,payment_value,ingestion_timestamp,source_file
b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
298fcdf1f73eb413e4d26d01b25bc1cd,1,credit_card,2,96.12,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
771ee386b001f06208a7419e4fc1bbd7,1,credit_card,1,81.16,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
3d7239c394a212faae122962df514ac7,1,credit_card,3,51.84,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
1f78449c87a54faf9e96e88ba1491fa9,1,credit_card,6,341.09,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv
0573b5e23cbd798006520e1d5b4c6714,1,boleto,1,51.95,2026-11-02T10:25:24.373Z,olist_order_payments_dataset.csv


In [0]:

# Check missing values in payments

payment_nulls = payments_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in payments_df.columns
])

display(payment_nulls)


order_id,payment_sequential,payment_type,payment_installments,payment_value,ingestion_timestamp,source_file
0,0,0,0,0,0,0


In [0]:

# Check duplicate payment keys

payment_duplicates = (
    payments_df
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Duplicate payment keys:",
    payment_duplicates.count()
)

display(payment_duplicates.limit(10))


Duplicate payment keys: 0


order_id,payment_sequential,count


In [0]:

# Convert payment columns to correct types

payments_clean = (
    payments_df
    .withColumn(
        "payment_sequential",
        F.expr("try_cast(payment_sequential AS INT)")
    )
    .withColumn(
        "payment_installments",
        F.expr("try_cast(payment_installments AS INT)")
    )
    .withColumn(
        "payment_value",
        F.expr("try_cast(payment_value AS DOUBLE)")
    )
)

payments_clean.printSchema()


root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# Standardize payment methods

payments_clean = payments_clean.withColumn(
    "payment_type",
    F.lower(F.trim(F.col("payment_type")))
)

display(
    payments_clean.groupBy("payment_type")
    .count()
    .orderBy(F.desc("count"))
)


payment_type,count
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


In [0]:

# Identify invalid payment records

invalid_payments = payments_clean.filter(
    F.col("payment_value").isNull()
    | (F.col("payment_value") < 0)
    | F.col("payment_sequential").isNull()
    | (F.col("payment_sequential") <= 0)
    | F.col("payment_installments").isNull()
    | (F.col("payment_installments") < 0)
)

print(
    "Invalid payment records:",
    invalid_payments.count()
)

display(invalid_payments.limit(10))


Invalid payment records: 0


order_id,payment_sequential,payment_type,payment_installments,payment_value,ingestion_timestamp,source_file


In [0]:

# Aggregate payments at order level

payments_summary = payments_clean.groupBy(
    "order_id"
).agg(
    F.round(
        F.sum("payment_value"), 2
    ).alias("total_payment_value"),

    F.count("*").alias("payment_count"),

    F.countDistinct(
        "payment_type"
    ).alias("payment_method_count")
)

display(payments_summary.limit(10))


order_id,total_payment_value,payment_count,payment_method_count
b81ef226f3fe1789b1e8b2acac839d17,99.33,1,1
a9810da82917af2d9aefd1278f1dcfa0,24.39,1,1
25e8ea4e93396b6fa0d3dd708e76c1bd,65.71,1,1
ba78997921bbcdc1373bb41e913ab953,107.78,1,1
42fdf880ba16b47b59251dd489d4441a,128.45,1,1
298fcdf1f73eb413e4d26d01b25bc1cd,96.12,1,1
771ee386b001f06208a7419e4fc1bbd7,81.16,1,1
3d7239c394a212faae122962df514ac7,51.84,1,1
1f78449c87a54faf9e96e88ba1491fa9,341.09,1,1
0573b5e23cbd798006520e1d5b4c6714,51.95,1,1


In [0]:

# Save cleaned payment records

(
    payments_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.order_payments")
)

print("Silver Order Payments table created!")


Silver Order Payments table created!


In [0]:

# Save order-level payment summary

(
    payments_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.payment_summary")
)

print("Payment Summary table created!")


Payment Summary table created!


In [0]:

# Verify Silver Payment tables

display(
    spark.sql("""
        SELECT
            'order_payments' AS table_name,
            COUNT(*) AS total_records
        FROM olist.silver.order_payments

        UNION ALL

        SELECT
            'payment_summary' AS table_name,
            COUNT(*) AS total_records
        FROM olist.silver.payment_summary
    """)
)


table_name,total_records
order_payments,103886
payment_summary,99440


In [0]:

# Payment method analysis

display(
    payments_clean.groupBy("payment_type")
    .agg(
        F.count("*").alias("transactions"),
        F.round(
            F.sum("payment_value"), 2
        ).alias("total_payment_value")
    )
    .orderBy(F.desc("total_payment_value"))
)


payment_type,transactions,total_payment_value
credit_card,76795,1.254208419E7
boleto,19784,2869361.27
voucher,5775,379436.87
debit_card,1529,217989.79
not_defined,3,0.0


In [0]:

from pyspark.sql import functions as F

# Load Bronze Geolocation dataset
geo_df = spark.table("olist.bronze.geolocation")

print("Total geolocation records:", geo_df.count())

display(geo_df.limit(10))


Total geolocation records: 1000163


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,ingestion_timestamp,source_file
01037,-23.54562128115268,-46.63929204800168,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01046,-23.546081127035535,-46.64482029837157,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01046,-23.54612896641469,-46.64295148361138,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01041,-23.5443921648681,-46.63949930627844,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01035,-23.541577961711493,-46.64160722329613,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01012,-23.547762303364266,-46.63536053788448,são paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01047,-23.546273112412678,-46.64122516971552,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01013,-23.546923208436723,-46.6342636964915,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01029,-23.543769055769133,-46.63427784085132,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv
01011,-23.547639550320632,-46.63603162315495,sao paulo,SP,2026-11-02T10:25:41.819Z,olist_geolocation_dataset.csv


In [0]:

# Check missing values

geo_nulls = geo_df.select([
    F.sum(
        F.col(c).isNull().cast("int")
    ).alias(c)
    for c in geo_df.columns
])

display(geo_nulls)


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,ingestion_timestamp,source_file
0,0,0,0,0,0,0


In [0]:

# Standardize geographical data

geo_clean = (
    geo_df
    .withColumn(
        "geolocation_zip_code_prefix",
        F.lpad(
            F.trim(F.col("geolocation_zip_code_prefix")),
            5,
            "0"
        )
    )
    .withColumn(
        "geolocation_lat",
        F.expr("try_cast(geolocation_lat AS DOUBLE)")
    )
    .withColumn(
        "geolocation_lng",
        F.expr("try_cast(geolocation_lng AS DOUBLE)")
    )
    .withColumn(
        "geolocation_city",
        F.lower(F.trim(F.col("geolocation_city")))
    )
    .withColumn(
        "geolocation_state",
        F.upper(F.trim(F.col("geolocation_state")))
    )
)

geo_clean.printSchema()


root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- source_file: string (nullable = true)



In [0]:

# Validate geographical coordinates

invalid_geo = geo_clean.filter(
    F.col("geolocation_lat").isNull()
    | F.col("geolocation_lng").isNull()
    | ~F.col("geolocation_lat").between(-90, 90)
    | ~F.col("geolocation_lng").between(-180, 180)
)

print(
    "Invalid coordinate records:",
    invalid_geo.count()
)

display(invalid_geo.limit(10))


Invalid coordinate records: 0


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,ingestion_timestamp,source_file


In [0]:

# Count exact duplicate geographic records

geo_columns = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state"
]

total_rows = geo_clean.count()

unique_rows = geo_clean.select(
    *geo_columns
).distinct().count()

print("Total records:", total_rows)
print("Unique geographical records:", unique_rows)
print("Exact duplicates:", total_rows - unique_rows)


Total records: 1000163
Unique geographical records: 738332
Exact duplicates: 261831


In [0]:

# Remove exact geographic duplicates

geo_deduplicated = geo_clean.dropDuplicates(
    [
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "geolocation_city",
        "geolocation_state"
    ]
)

print(
    "Records after deduplication:",
    geo_deduplicated.count()
)


Records after deduplication: 738332


In [0]:

# Retain valid coordinates for geographic aggregation

geo_valid = geo_deduplicated.filter(
    F.col("geolocation_zip_code_prefix").isNotNull()
    & F.col("geolocation_lat").between(-90, 90)
    & F.col("geolocation_lng").between(-180, 180)
)

# One representative coordinate per ZIP prefix
geo_summary = (
    geo_valid
    .groupBy("geolocation_zip_code_prefix")
    .agg(
        F.avg("geolocation_lat").alias("avg_latitude"),
        F.avg("geolocation_lng").alias("avg_longitude"),
        F.count("*").alias("location_count")
    )
)

display(geo_summary.limit(10))


geolocation_zip_code_prefix,avg_latitude,avg_longitude,location_count
01037,-23.54533514931352,-46.63899521271492,19
01046,-23.545884971576694,-46.643744209426345,37
01041,-23.544036423155266,-46.63988866258875,14
01035,-23.541612840173386,-46.64158346016764,15
01012,-23.54780758661154,-46.63484393782474,13
01047,-23.546124849824682,-46.64103782520949,18
01013,-23.547240037691147,-46.63422056848347,10
01029,-23.540574380051,-46.63312870430693,6
01011,-23.54664946252287,-46.63542523302689,4
01032,-23.53992249465614,-46.63513088832746,24


In [0]:

# Save cleaned geolocation data

(
    geo_deduplicated.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.geolocation")
)

print("Silver Geolocation table created!")


Silver Geolocation table created!


In [0]:

# Save ZIP-code summary

(
    geo_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.geolocation_summary")
)

print("Geolocation Summary table created!")


Geolocation Summary table created!


In [0]:

# Verify saved geolocation tables

display(
    spark.sql("""
        SELECT
            'geolocation' AS table_name,
            COUNT(*) AS total_records
        FROM olist.silver.geolocation

        UNION ALL

        SELECT
            'geolocation_summary' AS table_name,
            COUNT(*) AS total_records
        FROM olist.silver.geolocation_summary
    """)
)


table_name,total_records
geolocation,738332
geolocation_summary,19015


In [0]:

# Geolocation records by state

display(
    spark.table("olist.silver.geolocation")
    .groupBy("geolocation_state")
    .count()
    .orderBy(F.desc("count"))
)


geolocation_state,count
SP,285976
MG,101353
RJ,78836
RS,48093
PR,45059
SC,30191
BA,27720
GO,15601
PE,13162
ES,12632


In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Read corrected Bronze reviews
reviews_df = spark.table("olist.bronze.order_reviews")

print("Total Bronze reviews:", reviews_df.count())

display(
    reviews_df.select(
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    ).limit(10)
)


Total Bronze reviews: 99224


review_id,order_id,review_score,review_creation_date,review_answer_timestamp
7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,2018-01-18 00:00:00,2018-01-18 21:46:59
80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,2018-03-10 00:00:00,2018-03-11 03:05:13
228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,2018-02-17 00:00:00,2018-02-18 14:36:24
e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,2017-04-21 00:00:00,2017-04-21 22:02:06
f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,2018-03-01 00:00:00,2018-03-02 10:26:53
15197aa66ff4d0650b5434f1b46cda19,b18dcdf73be66366873cd26c5724d1dc,1,2018-04-13 00:00:00,2018-04-16 00:39:37
07f9bee5d1b850860defd761afa7ff16,e48aa0d2dcec3a2e87348811bcfdf22b,5,2017-07-16 00:00:00,2017-07-18 19:30:34
7c6400515c67679fbee952a7525281ef,c31a859e34e3adac22f376954e19b39d,5,2018-08-14 00:00:00,2018-08-14 21:36:06
a3f6f7f6f433de0aefbb97da197c554c,9c214ac970e84273583ab523dfafd09b,5,2017-05-17 00:00:00,2017-05-18 12:05:37
8670d52e15e00043ae7de4c01cc2fe06,b9bf720beb4ab3728760088589c62129,4,2018-05-22 00:00:00,2018-05-23 16:45:47


In [0]:

# Convert data types safely

reviews_clean = (
    reviews_df
    .withColumn(
        "review_score",
        F.expr("try_cast(review_score AS INT)")
    )
    .withColumn(
        "review_creation_date",
        F.to_timestamp("review_creation_date")
    )
    .withColumn(
        "review_answer_timestamp",
        F.to_timestamp("review_answer_timestamp")
    )
)

# Select latest review for each order
review_window = (
    Window.partitionBy("order_id")
    .orderBy(
        F.col("review_answer_timestamp").desc_nulls_last(),
        F.col("review_creation_date").desc_nulls_last(),
        F.col("review_id").desc()
    )
)

reviews_final = (
    reviews_clean
    .withColumn(
        "row_num",
        F.row_number().over(review_window)
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
    .select(
        "review_id",
        "order_id",
        "review_score",
        "review_creation_date",
        "review_answer_timestamp"
    )
)

print("Cleaned review records:", reviews_final.count())

display(reviews_final.limit(10))


Cleaned review records: 98673


review_id,order_id,review_score,review_creation_date,review_answer_timestamp
97ca439bc427b48bc1cd7177abe71365,00010242fe8c5a6d1ba2dd792cb16214,5,2017-09-21T00:00:00.000Z,2017-09-22T10:57:03.000Z
7b07bacd811c4117b742569b04ce3580,00018f77f2f0320c557190d7a144bdd3,4,2017-05-13T00:00:00.000Z,2017-05-15T11:34:13.000Z
0c5b33dea94867d1ac402749e5438e8b,000229ec398224ef6ca0657da4fc703e,5,2018-01-23T00:00:00.000Z,2018-01-23T16:06:31.000Z
f4028d019cb58564807486a6aaf33817,00024acbcdf0a6daa1e931b038114c75,4,2018-08-15T00:00:00.000Z,2018-08-15T16:39:01.000Z
940144190dcba6351888cafa43f3a3a5,00042b26cf59d7ce69dfabb4e55b4fd9,5,2017-03-02T00:00:00.000Z,2017-03-03T10:54:59.000Z
5e4e50af3b7960b7a10d86ec869509e8,00048cc3ae777c65dbb7d2a0634bc1ea,4,2017-05-23T00:00:00.000Z,2017-05-24T19:00:09.000Z
0381de7572d99d75230ce91207807265,00054e8431b9d7675808bcb819fb4a32,4,2017-12-19T00:00:00.000Z,2017-12-29T02:40:12.000Z
f0733ebeb51b637dfdc953983c867b89,000576fe39319847cbb9d288c5617fa6,5,2018-07-10T00:00:00.000Z,2018-07-11T20:14:25.000Z
67b1ab8f3ec19d5a90f153f21757d3a1,0005a1a1728c9d785b8e2b08b904576c,1,2018-03-30T00:00:00.000Z,2018-04-02T00:52:01.000Z
5c0b7e34ed85ec659bb064902d878e7a,0005f50442cb953dcd1d21e1fb923495,4,2018-07-05T00:00:00.000Z,2018-07-05T23:17:04.000Z


In [0]:

# Create the missing Silver Reviews table

(
    reviews_final.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist.silver.order_reviews")
)

print("SUCCESS: Silver Order Reviews table created!")


SUCCESS: Silver Order Reviews table created!


In [0]:

# Verify the table exists

print(
    "Table exists:",
    spark.catalog.tableExists(
        "olist.silver.order_reviews"
    )
)

print(
    "Total records:",
    spark.table(
        "olist.silver.order_reviews"
    ).count()
)


Table exists: True
Total records: 98673
